### Fine-Tuning a model to predict labels for LiDAR Point Clouds - 3D Semantic Segmentation - Waymo Open Dataset

#### By Jacob Igo

In [ ]:
import gcsfs
from google.cloud import storage
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import pandas as pd
import tensorflow as tf
import open3d as o3d
import numpy as np
import gc
import os
import random

import semseg_functions


In [ ]:

#get google cloud token
os.environ["CLOUDSDK_CONFIG"] = "/home/jacob/.config/gcloud"

import subprocess
token = subprocess.check_output(
    ["/usr/bin/gcloud", "auth", "print-access-token"]
).decode().strip()

from datetime import datetime, timezone, timedelta
fs = pafs.GcsFileSystem(access_token=token, credential_token_expiration=datetime.now(timezone.utc) + timedelta(hours=1))

folders = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/"))
lidar_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar/"))
seg_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar_segmentation/"))
calib_files = fs.get_file_info(pafs.FileSelector("waymo_open_dataset_v_2_0_0/training/lidar_calibration/"))


In [ ]:
def frame_loader(basefile, timestamp, sample_size=4096):

    calib_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar_calibration/{basefile}", filesystem=fs)
    calib_df = calib_pq.read_row_group(0).to_pandas()

    # seg_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar_segmentation/{basefile}", filesystem=fs)
    # lidar_pq = pq.ParquetFile(f"waymo_open_dataset_v_2_0_0/training/lidar/{basefile}", filesystem=fs)

    lidar_df = semseg_functions.timestamp_aligner(f"waymo_open_dataset_v_2_0_0/training/lidar/{basefile}", timestamp=timestamp)
    seg_df = semseg_functions.timestamp_aligner(f"waymo_open_dataset_v_2_0_0/training/lidar_segmentation/{basefile}", timestamp=timestamp)

    g_coords_1, masked_labels_1 = semseg_functions.laser_process(laser_num=1, df=lidar_df, df_rgc=calib_df, df_seg=seg_df, timestamp=timestamp)

    #sampling, as model expects fixed input size
    rng = np.random.default_rng(seed=42)
    random_indices = rng.integers(low=0, high=len(g_coords_1), size=sample_size)
    random_points = g_coords_1[random_indices]
    random_labels = masked_labels_1[random_indices]

    #normalizing, keeps points on a consistent scale for easier learning
    points_min = random_points.min(axis=0)
    points_max = random_points.max(axis=0)

    random_points_norm = (random_points - points_min) / (points_max - points_min)

    return random_points_norm, random_labels



